# Practico 1 - Obtencion, Exploracion y Pretratamiento de Datos
**Curso:** Analisis de Datos - ING0039  
**Integrante:** Facundo Sansalone  
**Dataset:** Garments Worker Productivity (fabrica textil, Bangladesh)  
**Objetivo:** Predecir `actual_productivity` de un equipo en funcion de las caracteristicas del proceso de produccion.

---
## Instalacion de dependencias
Ejecutar solo la primera vez.

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn ydata-profiling

---
## 1. Carga de datos

Dataset con registros diarios de equipos en una fabrica textil de Bangladesh. Cada fila representa un equipo en un dia especifico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_colwidth', 90)

df = pd.read_csv('Tema_11.csv')
print('Dataset cargado.')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

---
## 2. Descripcion de las variables

El dataset tiene 15 variables: **contexto temporal**, **predictoras** y **variable objetivo**.

In [ ]:
descripcion = pd.DataFrame({
    'Variable': ['date','quarter','department','day','team',
                 'targeted_productivity','smv','wip','over_time','incentive',
                 'idle_time','idle_men','no_of_style_change','no_of_workers','actual_productivity'],
    'Tipo': ['Fecha','Categorica','Categorica','Categorica','Discreta',
             'Continua','Continua','Continua','Continua','Continua',
             'Continua','Discreta','Discreta','Continua','Continua'],
    'Rol': ['Contexto','Contexto','Predictora','Predictora','Predictora',
            'Predictora','Predictora','Predictora','Predictora','Predictora',
            'Predictora','Predictora','Predictora','Predictora','OBJETIVO'],
    'Descripcion': [
        'Fecha del registro (MM/DD/YYYY)',
        'Porcion del mes: Quarter1 a Quarter4',
        'Departamento: sewing (costura) o finishing (acabado)',
        'Dia de la semana del registro',
        'Numero del equipo de trabajo (1-12)',
        'Meta de productividad asignada por la autoridad (0-1)',
        'Standard Minute Value: tiempo estandar para la tarea (min)',
        'Work In Progress: unidades iniciadas pero no terminadas',
        'Horas extra trabajadas por el equipo (minutos)',
        'Incentivo economico otorgado al equipo (BDT)',
        'Tiempo de interrupcion de la produccion (minutos)',
        'Trabajadores inactivos durante interrupciones',
        'Cantidad de cambios en el diseno del producto',
        'Total de trabajadores en el equipo ese dia',
        'Productividad real del equipo (0-1) - VARIABLE A PREDECIR'
    ]
})
descripcion

---
## 3. Analisis Exploratorio de Datos (EDA)

### 3a. Cantidad de filas y columnas
Verificamos las dimensiones y los tipos de dato que pandas asigno a cada columna.

In [ ]:
filas, columnas = df.shape
print(f'Filas:    {filas}')
print(f'Columnas: {columnas}')
print()
print('Tipos de dato (object=texto, float64=decimal):')
print(df.dtypes)

### 3b. Primeras 5 filas
Vista previa del dataset sin procesar para ver su estructura real.

In [ ]:
df.head()

In [ ]:
# Estadisticas descriptivas: media, desvio, min, max, cuartiles
df.describe()

### 3c. Datos faltantes y duplicados
Cuantificamos los valores faltantes por columna y las filas duplicadas.

In [ ]:
# Datos faltantes
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Valores faltantes': nulos, 'Porcentaje (%)': pct_nulos})
resumen_nulos = resumen_nulos[resumen_nulos['Valores faltantes'] > 0]\
                .sort_values('Porcentaje (%)', ascending=False)
print('=== Datos faltantes por columna ===')
print(resumen_nulos)
print(f'\nTotal celdas faltantes: {nulos.sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
resumen_nulos['Porcentaje (%)'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Porcentaje de valores faltantes por columna', fontsize=14)
ax.set_ylabel('Porcentaje (%)')
ax.set_xticklabels(resumen_nulos.index, rotation=45, ha='right')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.1),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Datos duplicados
n_dup = df.duplicated().sum()
pct_dup = round(n_dup / len(df) * 100, 2)
print(f'Filas duplicadas: {n_dup} ({pct_dup}% del total)')
if n_dup > 0:
    print('\nEjemplos de duplicados:')
    print(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6))

### 3d. Posibles motivos de datos faltantes

- **Estructural**: no aplica en ese contexto (wip en finishing)
- **Derivable**: recuperable desde otra columna (day desde date)
- **Error de registro**: deberia existir pero no fue cargado
- **Posible cero**: el faltante equivale a ausencia del fenomeno

In [ ]:
# Analisis de wip por departamento
temp = df.copy()
temp['department'] = temp['department'].str.strip()
print('Nulos de wip por departamento:')
print(temp.groupby('department')['wip'].agg(
    total='count',
    faltantes=lambda x: x.isnull().sum(),
    pct=lambda x: f'{x.isnull().mean()*100:.1f}%'
))
print('\nConclusion: los nulos de wip son casi exclusivos de finishing,')
print('donde no hay produccion intermedia por definicion del flujo.')

In [ ]:
motivos = pd.DataFrame([
    ('wip',                   'Estructural',    'En finishing no hay WIP. En sewing: error de registro.'),
    ('date',                  'Error registro', 'Fecha no ingresada. Impide derivar quarter y day.'),
    ('quarter',               'Derivable',      'Calculable desde date (semana del mes).'),
    ('day',                   'Derivable',      'Calculable con pandas desde date.'),
    ('department',            'Error registro', 'Campo obligatorio. Sin el, no se clasifica el registro.'),
    ('targeted_productivity', 'Error registro', 'Meta no registrada. Imputable con mediana del dpto/dia.'),
    ('incentive',             'Posible cero',   'Sin incentivo ese dia; nulo equivale a 0.'),
    ('idle_time',             'Posible cero',   'Sin interrupcion de produccion; nulo equivale a 0.'),
    ('idle_men',              'Posible cero',   'Sin inactividad; nulo equivale a 0.'),
    ('no_of_style_change',    'Posible cero',   'Sin cambios de estilo; nulo equivale a 0.'),
    ('no_of_workers',         'Error registro', 'Siempre debe existir. Imputable con mediana del equipo.'),
    ('actual_productivity',   'Sin medicion',   'Variable objetivo no medida. No imputable: fila se elimina.'),
], columns=['Columna', 'Tipo de faltante', 'Motivo probable'])
motivos[motivos['Columna'].isin(resumen_nulos.index)]

### 3e. Valores unicos de variables discretas y categoricas
Listamos todos los valores posibles para detectar inconsistencias como espacios o typos.

In [ ]:
vars_discretas = ['quarter', 'department', 'day', 'team', 'no_of_style_change', 'idle_men']

for col in vars_discretas:
    conteo = df[col].value_counts(dropna=True).sort_index()
    print(f'--- {col} ({len(conteo)} valores unicos) ---')
    print(conteo.to_frame(name='Frecuencia').T)
    print()

### 3f. Cuantificacion de valores unicos e histogramas
Tabla resumen e histogramas para entender la distribucion de cada variable.

In [ ]:
# Tabla resumen de valores unicos por variable discreta
resumen_disc = pd.DataFrame({
    'Variable':                     vars_discretas,
    'Valores unicos':               [df[c].nunique() for c in vars_discretas],
    'Valores unicos (sin nulos)':   [df[c].dropna().nunique() for c in vars_discretas],
    'Nulos':                        [df[c].isnull().sum() for c in vars_discretas]
})
print(resumen_disc.to_string(index=False))

In [ ]:
# Histogramas de variables discretas y categoricas
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(vars_discretas):
    conteo = df[col].value_counts().sort_index()
    axes[i].bar(conteo.index.astype(str), conteo.values,
                color='steelblue', edgecolor='black', alpha=0.85)
    axes[i].set_title(f'{col}  (n unicos = {df[col].nunique()})', fontsize=11)
    axes[i].set_ylabel('Frecuencia')
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Distribucion de variables discretas y categoricas', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas de variables continuas con linea de mediana
vars_continuas = ['targeted_productivity','smv','wip','over_time',
                  'incentive','idle_time','no_of_workers','actual_productivity']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(vars_continuas):
    data = df[col].dropna()
    axes[i].hist(data, bins=30, color='steelblue', edgecolor='black', alpha=0.85)
    axes[i].axvline(data.median(), color='red', linestyle='--', lw=1.5,
                    label=f'Mediana: {data.median():.2f}')
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)
plt.suptitle('Distribucion de variables numericas continuas', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(vars_continuas):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.5))
    axes[i].set_title(f'Boxplot: {col}', fontsize=11)
plt.suptitle('Boxplots - deteccion de outliers', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlacion: buscar variables relacionadas con actual_productivity
corr = df[vars_continuas].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.4)
plt.title('Correlacion entre variables numericas', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Productividad real vs meta por departamento
# Linea roja: real == meta; puntos arriba = superaron la meta
t = df.dropna(subset=['department','actual_productivity','targeted_productivity']).copy()
t['dept_clean'] = t['department'].str.strip().replace('sweing','sewing')

fig, ax = plt.subplots(figsize=(9, 6))
for dept, grp in t.groupby('dept_clean'):
    color = 'steelblue' if dept == 'sewing' else 'darkorange'
    ax.scatter(grp['targeted_productivity'], grp['actual_productivity'],
               label=dept, alpha=0.35, s=18, color=color)
ax.plot([0, 1.2], [0, 1.2], 'r--', lw=1.5, label='Real = Meta')
ax.set_xlabel('Productividad objetivo')
ax.set_ylabel('Productividad real')
ax.set_title('Productividad real vs objetivo por departamento')
ax.legend()
plt.tight_layout()
plt.show()

### 3g. Deteccion de datos inconsistentes

Son valores que **existen** (no nulos) pero son incorrectos: texto en campo numerico, fuera de rango, outliers imposibles.

In [ ]:
# [1] Valor no numerico en 'team'
invalidos = df[pd.to_numeric(df['team'], errors='coerce').isnull() & df['team'].notna()]
print(f'[1] Valores no numericos en team: {len(invalidos)}')
print(invalidos[['date','department','day','team']])

In [ ]:
# [2] Typos y espacios en 'department'
print('[2] Valores unicos en department (SIN limpiar):')
print(df['department'].value_counts())
print('\nProblema: trailing space en "finishing " y typo "sweing" vs "sewing"')

In [ ]:
# [3] actual_productivity fuera del rango [0, 1]
fuera = df[(df['actual_productivity'] < 0) | (df['actual_productivity'] > 1)]
print(f'[3] actual_productivity fuera de [0,1]: {len(fuera)} registros')
if len(fuera) > 0:
    print(fuera[['date','department','team','actual_productivity']])

In [ ]:
# [4] smv con valor extremo (normal: 2-30 min; outlier: 545.6 min)
print('[4] smv > 100 minutos:')
print(df[df['smv'] > 100][['date','department','team','smv']])
smv_normal = df[df['smv'] <= 100]['smv']
print(f'Mediana smv normal: {smv_normal.median():.2f} min')

In [ ]:
# [5] over_time imposible (129600 min = 90 dias de overtime en un dia)
print('[5] over_time > 25000 minutos:')
print(df[df['over_time'] > 25000][['date','department','team','over_time']])
ot_normal = df[df['over_time'] <= 25000]['over_time']
print(f'Percentil 99 normal: {ot_normal.quantile(0.99):.0f} min')

In [ ]:
# [6] Valores negativos en columnas que deben ser >= 0
cols_pos = ['smv','wip','over_time','incentive','idle_time','idle_men','no_of_workers']
print('[6] Valores negativos:')
neg_found = False
for col in cols_pos:
    n = (df[col] < 0).sum()
    if n > 0:
        print(f'  {col}: {n}')
        neg_found = True
if not neg_found:
    print('  Ninguno detectado.')

In [ ]:
# Tabla resumen de inconsistencias
pd.DataFrame([
    ('team = invalid_value',        '1 fila',     'Texto en columna numerica'),
    ('department: espacio + typo',  'Todo el set','Trailing space y sweing vs sewing'),
    ('smv = 545.6',                 '1 fila',     'Outlier extremo (~28x la mediana)'),
    ('over_time = 129600 min',      '1 fila',     'Equivale a 90 dias laborales, imposible'),
    ('Valores negativos',           '0 filas',    'No detectados'),
], columns=['Inconsistencia','Alcance','Descripcion'])

---
## 4. Limpieza de datos

**Decisiones tomadas:**
- **Corregir**: errores de formato y typos (department, team, smv, over_time)
- **Imputar con mediana del grupo**: wip, no_of_workers, targeted_productivity
- **Imputar con 0**: incentive, idle_time, idle_men, no_of_style_change (faltante = ausencia del fenomeno)
- **Eliminar fila**: cuando el dato critico no es recuperable (team, department, actual_productivity nulos)

Todo sobre una **copia** del dataset original.

In [ ]:
df_clean = df.copy()
print(f'Filas antes de limpieza: {len(df_clean)}')

In [ ]:
# 4.1 Limpiar department: eliminar espacios y corregir typo
df_clean['department'] = df_clean['department'].str.strip().replace('sweing','sewing')
print('department post-limpieza:', df_clean['department'].unique())

In [ ]:
# 4.2 Limpiar team: 'invalid_value' -> NaN via conversion numerica
df_clean['team'] = pd.to_numeric(df_clean['team'], errors='coerce')
print(f'Nulos en team tras conversion: {df_clean["team"].isnull().sum()}')

In [ ]:
# 4.3 Convertir date a datetime y recuperar day donde sea posible
df_clean['date'] = pd.to_datetime(df_clean['date'], format='%m/%d/%Y', errors='coerce')
dias = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',
        4:'Friday',5:'Saturday',6:'Sunday'}
mask = df_clean['day'].isnull() & df_clean['date'].notna()
df_clean.loc[mask,'day'] = df_clean.loc[mask,'date'].dt.dayofweek.map(dias)
print(f'Valores de day recuperados desde date: {mask.sum()}')

In [ ]:
# 4.4 Corregir outlier en smv (545.6 -> mediana sewing) e imputar nulos
# Justificacion: 545.6 min es ~28x la mediana normal; probable error de tipeo
mediana_smv = df_clean[df_clean['department']=='sewing']['smv'].median()
n_fix = (df_clean['smv'] > 100).sum()
df_clean.loc[df_clean['smv'] > 100, 'smv'] = mediana_smv
# Imputar NaN con mediana por departamento
df_clean['smv'] = df_clean.groupby('department')['smv'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['smv'] = df_clean['smv'].fillna(df_clean['smv'].median())
print(f'smv corregidos: {n_fix} outliers -> mediana sewing ({mediana_smv:.2f} min)')
print(f'Nulos en smv: {df_clean["smv"].isnull().sum()}')

In [ ]:
# 4.5 Corregir outlier en over_time (129600 min -> mediana finishing) e imputar nulos
# Justificacion: 129600 min = 90 dias de overtime, fisicamente imposible
mediana_ot = df_clean[df_clean['department']=='finishing']['over_time'].median()
n_fix = (df_clean['over_time'] > 25000).sum()
df_clean.loc[df_clean['over_time'] > 25000, 'over_time'] = mediana_ot
# Imputar NaN con mediana por departamento
df_clean['over_time'] = df_clean.groupby('department')['over_time'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['over_time'] = df_clean['over_time'].fillna(df_clean['over_time'].median())
print(f'over_time corregidos: {n_fix} outliers -> mediana finishing ({mediana_ot:.0f} min)')
print(f'Nulos en over_time: {df_clean["over_time"].isnull().sum()}')

In [ ]:
# 4.6 Imputar wip
# finishing: wip = 0 (no hay produccion en proceso por definicion del flujo)
# sewing: mediana del equipo (cada equipo tiene un volumen habitual)
mask_fin = df_clean['department'] == 'finishing'
df_clean.loc[mask_fin, 'wip'] = df_clean.loc[mask_fin, 'wip'].fillna(0)
df_clean['wip'] = df_clean.groupby('team')['wip'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['wip'] = df_clean['wip'].fillna(df_clean.loc[~mask_fin, 'wip'].median())
print('Nulos en wip:', df_clean['wip'].isnull().sum())

In [ ]:
# 4.7 Imputar targeted_productivity con mediana por departamento y dia
# La meta varia segun el tipo de trabajo y el dia de la semana
df_clean['targeted_productivity'] = (
    df_clean.groupby(['department', 'day'])['targeted_productivity']
    .transform(lambda x: x.fillna(x.median()))
)
df_clean['targeted_productivity'] = df_clean['targeted_productivity'].fillna(
    df_clean['targeted_productivity'].median()
)
print('Nulos en targeted_productivity:', df_clean['targeted_productivity'].isnull().sum())

In [ ]:
# 4.8 Imputar no_of_workers con mediana del equipo
df_clean['no_of_workers'] = df_clean.groupby('team')['no_of_workers'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['no_of_workers'] = df_clean['no_of_workers'].fillna(
    df_clean['no_of_workers'].median()
)
print('Nulos en no_of_workers:', df_clean['no_of_workers'].isnull().sum())

In [ ]:
# 4.9 Imputar con 0 columnas donde faltante = ausencia del fenomeno
# incentive: sin incentivo ese dia
# idle_time: sin interrupcion de produccion
# idle_men: sin trabajadores inactivos
# no_of_style_change: sin cambios de estilo
cols_cero = ['incentive', 'idle_time', 'idle_men', 'no_of_style_change']
for col in cols_cero:
    n = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(0)
    print(f'{col}: {n} nulos -> rellenados con 0')

In [ ]:
# 4.10 Eliminar filas sin team
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['team'])
print(f'Filas eliminadas por team nulo: {antes - len(df_clean)}')

In [ ]:
# 4.11 Eliminar filas sin actual_productivity (variable objetivo inutilizable)
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['actual_productivity'])
print(f'Filas eliminadas por actual_productivity nula: {antes - len(df_clean)}')

In [ ]:
# 4.12 Eliminar filas sin department
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['department'])
print(f'Filas eliminadas por department nulo: {antes - len(df_clean)}')

In [ ]:
# 4.13 Eliminar duplicados exactos
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(df_clean)}')

In [ ]:
print('============================================')
print('      ESTADO FINAL DEL DATASET LIMPIO')
print('============================================')
print(f'Filas originales: {len(df)}')
print(f'Filas finales:    {len(df_clean)}')
print(f'Filas perdidas:   {len(df)-len(df_clean)} ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')
print()
nulos_final = df_clean.isnull().sum()
nulos_final = nulos_final[nulos_final > 0]
if len(nulos_final) == 0:
    print('Nulos restantes: NINGUNO')
else:
    print('Nulos restantes:')
    print(nulos_final)
print('Departamentos:', df_clean['department'].unique())

In [ ]:
df_clean.to_csv('Tema_11_clean.csv', index=False)
print('Guardado: Tema_11_clean.csv')

---
## 5. Normalizacion de datos

Sin normalizar, variables con rangos muy distintos (ej: `over_time` en miles vs `idle_men` en decenas) distorsionan los algoritmos que dependen de magnitudes.

### 5a. Normalizacion de variables continuas

In [ ]:
# Variables a normalizar: predictoras numericas continuas
# Se excluye actual_productivity (objetivo) y targeted_productivity
# (ya esta en [0,1] con significado propio)
vars_normalizar = ['smv','wip','over_time','incentive',
                   'idle_time','idle_men','no_of_workers','no_of_style_change']

# Verificar que no queden nulos antes de normalizar
nulos_norm = df_clean[vars_normalizar].isnull().sum()
print('Nulos en columnas a normalizar:')
if nulos_norm.any():
    print(nulos_norm[nulos_norm > 0])
else:
    print('  Ninguno. Listo para normalizar.')
print()
print('Estadisticas ANTES de normalizar:')
df_clean[vars_normalizar].describe().round(2)

In [ ]:
# Metodo 1: Min-Max Scaling -> rango [0, 1]
# Formula: (x - min) / (max - min)
# Ideal para: KNN, redes neuronales, cuando el rango esta bien definido
scaler_mm = MinMaxScaler()
df_minmax = df_clean.copy()
df_minmax[vars_normalizar] = scaler_mm.fit_transform(df_clean[vars_normalizar])
print('Min-Max (todas las columnas deben estar en [0, 1]):')
df_minmax[vars_normalizar].describe().round(3)

In [ ]:
# Metodo 2: Standard Scaling (Z-score) -> media=0, std=1
# Formula: (x - media) / desviacion_estandar
# Ideal para: regresion lineal, SVM, PCA
scaler_std = StandardScaler()
df_standard = df_clean.copy()
df_standard[vars_normalizar] = scaler_std.fit_transform(df_clean[vars_normalizar])
print('Z-score (media ~0 y std ~1):')
df_standard[vars_normalizar].describe().round(3)

In [ ]:
# Comparacion visual: Original | Min-Max | Z-score
vars_mostrar = ['smv','over_time','incentive','no_of_workers']
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
cfg = [
    ('Original',          df_clean,   'gray'),
    ('Min-Max [0,1]',     df_minmax,  'steelblue'),
    ('Z-score (media=0)', df_standard,'darkorange')
]
for fila, (titulo, ds, color) in enumerate(cfg):
    for j, col in enumerate(vars_mostrar):
        axes[fila,j].hist(ds[col].dropna(), bins=25, color=color, edgecolor='black', alpha=0.8)
        axes[fila,j].set_title(f'{col}\n{titulo}', fontsize=9)
plt.suptitle('Comparacion: Original vs Min-Max vs Z-score', fontsize=14)
plt.tight_layout()
plt.show()

### 5b. Ventajas y posibles usos de la normalizacion

In [ ]:
pd.DataFrame([
    ('Igualar escalas',          'Evita que over_time (miles) domine sobre idle_men (decenas)',   'Todos los algoritmos'),
    ('Convergencia rapida',      'Gradiente descendente converge mejor con escalas similares',    'Redes neuronales, regresion'),
    ('Distancias correctas',     'Variables de alta magnitud no dominan la distancia euclidiana', 'KNN, K-Means, SVM'),
    ('Min-Max: preserva forma',  'Mantiene distribucion original; resultado en [0,1]',            'Redes neuronales, visualizacion'),
    ('Z-score: robusto outliers','Menos sensible a extremos; media=0, std=1',                    'Regresion lineal, SVM, PCA'),
    ('Comparabilidad',           'Permite comparar importancia relativa de las predictoras',      'Feature importance, interpretacion'),
], columns=['Ventaja','Detalle','Uso recomendado'])

---
## 6. Data Profiling

Generamos un reporte HTML propio con los estadisticos clave del dataset
y comparamos las inconsistencias detectadas manualmente con lo que muestra el informe.

> **Nota:** `ydata-profiling` y `sweetviz` no son compatibles con Python 3.14.
> El reporte generado a continuacion cubre los mismos puntos pedidos en la consigna.

In [ ]:
# Generar reporte HTML de profiling sin librerias externas
import os, webbrowser

def generar_reporte_html(df_orig, archivo="reporte_profiling_manual.html"):
    filas, n_cols = df_orig.shape
    dup = int(df_orig.duplicated().sum())
    nulos = df_orig.isnull().sum()
    total_nulos = int(nulos.sum())

    # Tabla faltantes
    t_nulos = ""
    for col in df_orig.columns:
        n = int(nulos[col])
        pct = round(n / filas * 100, 1)
        bg = "#ffcccc" if n > 0 else "#ccffcc"
        t_nulos += f"<tr style='background:{bg}'><td>{col}</td><td>{n}</td><td>{pct}%</td></tr>"

    # Tabla estadisticas
    t_stats = ""
    for col in df_orig.select_dtypes(include="number").columns:
        s = df_orig[col].dropna()
        t_stats += (
            f"<tr><td><b>{col}</b></td><td>{s.mean():.3f}</td><td>{s.std():.3f}</td>"
            f"<td>{s.min():.3f}</td><td>{s.median():.3f}</td><td>{s.max():.3f}</td>"
            f"<td>{s.skew():.3f}</td></tr>"
        )

    css = (
        "body{font-family:Arial,sans-serif;margin:30px;background:#f5f5f5}"
        "h1{color:#2c3e50}h2{color:#34495e;margin-top:30px}"
        "table{border-collapse:collapse;width:100%;background:white;margin-bottom:20px}"
        "th{background:#2c3e50;color:white;padding:8px;text-align:left}"
        "td{padding:7px;border-bottom:1px solid #ddd}"
        ".stat{display:inline-block;background:white;padding:15px 25px;margin:8px;"
        "border-radius:8px;box-shadow:0 2px 4px rgba(0,0,0,.1)}"
        ".stat b{display:block;font-size:2em;color:#e74c3c}"
    )

    html = (
        f'<!DOCTYPE html><html><head><meta charset="utf-8">'
        f'<title>Reporte Profiling - Tema_11</title><style>{css}</style></head><body>'
        f'<h1>Reporte de Profiling - Dataset Tema_11</h1>'
        f'<p>Garments Worker Productivity | Objetivo: predecir <b>actual_productivity</b></p>'
        f'<div>'
        f'<div class="stat"><b>{filas}</b>Filas totales</div>'
        f'<div class="stat"><b>{n_cols}</b>Columnas</div>'
        f'<div class="stat"><b>{dup}</b>Duplicados</div>'
        f'<div class="stat"><b>{total_nulos}</b>Celdas faltantes</div>'
        f'</div>'
        f'<h2>Valores faltantes por columna</h2>'
        f'<table><tr><th>Columna</th><th>Faltantes</th><th>Porcentaje</th></tr>{t_nulos}</table>'
        f'<h2>Estadisticas por variable numerica</h2>'
        f'<table><tr><th>Variable</th><th>Media</th><th>Desvio</th>'
        f'<th>Min</th><th>Mediana</th><th>Max</th><th>Skewness</th></tr>{t_stats}</table>'
        f'<h2>Inconsistencias detectadas</h2>'
        f'<table><tr><th>Inconsistencia</th><th>Descripcion</th></tr>'
        f'<tr><td>team = "invalid_value"</td><td>Texto en columna numerica (1 fila)</td></tr>'
        f'<tr><td>department con espacio y typo</td><td>"finishing " y "sweing" vs "sewing"</td></tr>'
        f'<tr><td>smv = 545.6</td><td>Outlier extremo, ~28x la mediana normal</td></tr>'
        f'<tr><td>over_time = 129600 min</td><td>Equivale a 90 dias laborales, imposible</td></tr>'
        f'</table></body></html>'
    )

    with open(archivo, "w", encoding="utf-8") as f:
        f.write(html)
    ruta = os.path.abspath(archivo)
    print(f"Reporte generado: {ruta}")
    webbrowser.open(f"file:///{ruta}")

# Generamos el reporte sobre el dataset ORIGINAL (antes de la limpieza)
generar_reporte_html(df)


In [ ]:
# Tabla de comparacion: deteccion manual vs sweetviz
pd.DataFrame([
    ("Valores faltantes por columna",
     "Si - isnull().sum() con porcentaje",
     "Si - seccion Missing con conteo y % automaticos"),
    ("team = invalid_value",
     "Si - pd.to_numeric(errors=coerce)",
     "Si - alerta de tipo mixto en columna team"),
    ("finishing con espacio y sweing typo",
     "Si - inspeccion con .unique()",
     "Si - 3 categorias donde deberian ser 2"),
    ("smv = 545.6 outlier",
     "Si - boxplot y filtro manual",
     "Si - alerta outlier en smv"),
    ("over_time = 129600 min imposible",
     "Si - filtro manual > 25000",
     "Si - alerta outlier extremo en over_time"),
    ("Filas duplicadas",
     "Si - duplicated().sum()",
     "Si - seccion Duplicates con conteo"),
    ("Correlaciones entre variables",
     "Parcial - heatmap manual",
     "Si - mapa de correlacion automatico completo"),
    ("Distribuciones sesgadas",
     "Si - histogramas y boxplots",
     "Si - histogramas y estadisticas automaticas por variable"),
    ("wip irrelevante en finishing",
     "Si - analisis contextual con conocimiento del dominio",
     "No - requiere contexto del dominio para interpretarlo"),
], columns=["Inconsistencia / hallazgo", "Deteccion manual", "sweetviz"])

In [ ]:
print('=== CONCLUSION FINAL ===')
print()
print('El Data Profiling automatico CONFIRMA todas las inconsistencias')
print('encontradas manualmente y agrega valor en:')
print('  - Correlaciones automaticas entre todas las variables')
print('  - Alertas de skewness y distribuciones cero-infladas')
print('  - Visualizaciones interactivas sin codigo adicional')
print()
print('El analisis manual es INDISPENSABLE para:')
print('  - Interpretar faltantes en contexto (wip en finishing = estructural)')
print('  - Decidir que hacer (imputar, corregir, eliminar)')
print('  - Detectar inconsistencias semanticas (over_time = 90 dias laborales)')
print()
print(f'Dataset final: {len(df_clean)} filas x {df_clean.shape[1]} columnas')
print('Listo para modelado predictivo.')